# C11-neural-training — Practice p08 — Solution


**Type:** constrained coding · **Difficulty:** core · **Concepts:** autograd-training, torch-optimizers


Autograd and the hand calculation use the same pre-step snapshot. Only after
both gradients are cloned does SGD own the update.


In [ ]:
import torch
import torch.nn as nn

def autograd_and_optimizer_step(X, y, w0, learning_rate):
    X = X.detach().clone().to(dtype=torch.float64, device="cpu")
    y = y.detach().clone().to(dtype=torch.float64, device="cpu")
    initial = w0.detach().clone().to(dtype=torch.float64, device="cpu")
    w = nn.Parameter(initial.clone())
    optimizer = torch.optim.SGD([w], lr=learning_rate)
    optimizer.zero_grad(set_to_none=True)
    loss = ((X @ w - y) ** 2).mean()
    loss.backward()
    autograd_grad = w.grad.detach().clone()
    hand_grad = (2.0 / X.shape[0]) * X.T @ (X @ initial - y)
    before = w.detach().clone()
    optimizer.step()
    after = w.detach().clone()
    return {"autograd_grad": autograd_grad, "hand_grad": hand_grad.detach().clone(),
            "before": before, "after": after, "loss_before": float(loss.detach()),
            "loss_after": float(((X @ after-y)**2).mean())}

X_p08 = torch.tensor([[1., 2.], [-1., 1.], [2., -1.]], dtype=torch.float64)
y_p08 = torch.tensor([1., -0.5, 2.], dtype=torch.float64)
w_p08 = torch.tensor([0.2, -0.1], dtype=torch.float64)
result_p08 = autograd_and_optimizer_step(X_p08, y_p08, w_p08, 0.05)


### Answer check


In [ ]:
assert torch.allclose(result_p08["autograd_grad"], result_p08["hand_grad"], atol=1e-11, rtol=1e-10)
expected_after_p08 = w_p08 - 0.05 * result_p08["hand_grad"]
assert torch.allclose(result_p08["after"], expected_after_p08, atol=1e-11, rtol=1e-10)
assert torch.equal(result_p08["before"], w_p08) and not torch.equal(result_p08["after"], w_p08)
assert result_p08["loss_after"] < result_p08["loss_before"]
